# 09 · Accelerate — 1×N GPU

`accelerate launch --multi_gpu --num_processes N torch_distributor_trainer.py ...`. driver의 N개 A10G에 N개 프로세스를 띄운다.

> 1×1은 [`08-launch_accelerator_1x1.ipynb`](08-launch_accelerator_1x1.ipynb), M×N은 [`10-launch_accelerator_MxN.ipynb`](10-launch_accelerator_MxN.ipynb).

**클러스터**: Single node, DBR 17.3 LTS ML, Driver `g5.12xlarge` (4× A10G), Workers 0.

In [ ]:
%run ./00-setup

## 경로

In [ ]:
import os

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
TRAINER_SCRIPT = os.path.join(SCRIPT_DIR, "torch_distributor_trainer.py")
print(f"SCRIPT_DIR={SCRIPT_DIR}")
print(f"TRAINER_SCRIPT={TRAINER_SCRIPT}")

## 1×N GPU

In [ ]:
import subprocess
import mlflow

NUM_GPUS_PER_NODE = 4  # 클러스터 driver의 GPU 수

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="acc-1xN", log_system_metrics=True) as run:
    run_id = run.info.run_id

cmd = [
    "accelerate", "launch",
    "--multi_gpu",
    "--num_processes", str(NUM_GPUS_PER_NODE),
    TRAINER_SCRIPT,
    "--run_id", run_id,
    "--db_host", DB_HOST,
    "--db_token", DB_TOKEN,
    "--data_dir", DATA_DIR,
    "--ckpt_path", os.path.join(CKPT_DIR, "acc-1xN.pt"),
    "--n_users", str(cfg["n_users"]),
    "--n_items", str(cfg["n_items"]),
    "--emb_dim", str(cfg["emb_dim"]),
    "--tower_hidden", *[str(x) for x in cfg["tower_hidden"]],
    "--batch_size", str(cfg["batch_size_per_gpu"]),
    "--num_epochs", str(cfg["num_epochs"]),
    "--max_steps_per_epoch", str(cfg["max_steps_per_epoch"]),
    "--patience", str(PATIENCE),
    "--min_delta", str(MIN_DELTA),
    "--topology", "1xN",
    "--script_dir", SCRIPT_DIR,
]
print("running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=SCRIPT_DIR)